# VIX Signal Sensitivity Near the Close

#### **MScFE 690 Capstone -Group 17843**

**Authors**: Edgar NAVA-edgar.nava@grovewm.com & Celestin NYANDWI-celestinnyandwi48@gmail.com

This notebook reproduces the Monte Carlo sensitivity analysis in Appendix C. We apply Model 2073 to synthetic SVXY −1x in one continuous simulation over Development and Validation (October 4, 2011–December 29, 2023), excluding Test.

## 1. Data and model settings

We use the same rules, costs and performance measures as `08-Model_2073_SVXY-ONEx.ipynb`. The inputs are `VIX_data.csv` in `data/raw/` and `SVXY_synth-ONEx.csv` in `data/processed/`, using the paths in `config.json`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project folder.")
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))
RAW_DATA_DIR = PROJECT_ROOT / CONFIG["raw_data_dir"]
PROCESSED_DATA_DIR = PROJECT_ROOT / CONFIG["processed_data_dir"]

START, END = "2011-10-04", "2023-12-29"
N_SIMULATIONS, SEED, INTERVALS = 200, 2073, 26
INITIAL_CASH = 100_000.0
COMMISSION_PER_SHARE = 0.005
SLIPPAGE_PCT = 0.001
SESSIONS_PER_YEAR = 252
ANNUAL_RISK_FREE_RATE = 0.0
STATE_THRESHOLDS = [17.5, 22.540001, 28.805999000000003, 37.810001]
RULES = {
    "C2": {"vix_min": 23.0, "cash_fraction": 0.15,
           "target_nav": 0.70, "trailing_stop_pct": 0.20},
    "C3": {"vix_max": 28.0, "cash_fraction": 0.10,
           "invested_cash_fraction": 0.50, "trailing_stop_pct": 0.40},
    "V2": {"vix_min": 19.0},
    "V3": {"vix_max": 15.0},
}

vix = pd.read_csv(RAW_DATA_DIR / "VIX_data.csv", parse_dates=["Date"]).set_index("Date").sort_index().loc[:END, ["High", "Low", "Close"]]
etf = pd.read_csv(PROCESSED_DATA_DIR / "SVXY_synth-ONEx.csv", parse_dates=["Date"]).set_index("Date").sort_index().loc[START:END, ["Close"]]
# Retain one preceding observation for the first session's previous state.
previous_date = vix.index[vix.index < pd.Timestamp(START)][-1]
vix = vix.loc[previous_date:END]
if (vix.isna().any().any() or etf.isna().any().any()
        or not vix.index.is_unique or not etf.index.is_unique
        or (vix.High < vix.Low).any()):
    raise ValueError("Check the input dates and required VIX/SVXY observations.")

original_close = vix.Close.to_numpy()
amplitude = (vix.High - vix.Low).to_numpy() / INTERVALS

def aligned_prices(close):
    signals = pd.DataFrame({"VIX_Close": close}, index=vix.index)
    signals["State"] = np.searchsorted(STATE_THRESHOLDS, close, side="left") + 1
    signals["Previous_state"] = signals.State.shift(1)
    data = etf.join(signals, how="left")
    if data.empty or data.isna().any().any():
        raise ValueError("Missing aligned observations.")
    return data


## 2. Model 2073

Each account starts with USD 100,000 in cash and continues through Validation without a reset. The rule thresholds, state boundaries, allocation parameters, trailing-stop percentages, costs and ETF Close prices remain unchanged.

In [2]:
def finish_account(account):
    account = account.copy()
    account["Position_value"] = account.Units * account.Close
    account["Equity"] = account.Cash + account.Position_value
    account["Daily_return"] = account.Equity.pct_change(fill_method=None)
    account.loc[account.index[0], "Daily_return"] = account.Equity.iloc[0] / INITIAL_CASH - 1
    account["Drawdown"] = account.Equity / account.Equity.cummax().clip(lower=INITIAL_CASH) - 1
    account["Exposure"] = account.Position_value / account.Equity
    return account

def performance(account):
    final = float(account.Equity.iloc[-1])
    years = (account.index[-1] - account.index[0]).days / 365.25
    cagr = (final / INITIAL_CASH) ** (1 / years) - 1
    max_drawdown = float(account.Drawdown.min())
    daily_rf = (1 + ANNUAL_RISK_FREE_RATE) ** (1 / SESSIONS_PER_YEAR) - 1
    excess = account.Daily_return - daily_rf
    volatility = float(excess.std(ddof=1))
    return pd.Series({
        "Initial equity ($)": INITIAL_CASH,
        "Final equity ($)": final,
        "Total profit ($)": final - INITIAL_CASH,
        "Total return": final / INITIAL_CASH - 1,
        "Annualized return (CAGR)": cagr,
        "Sharpe ratio": np.sqrt(SESSIONS_PER_YEAR) * excess.mean() / volatility
            if volatility > 0 else np.nan,
        "Calmar ratio": cagr / abs(max_drawdown)
            if np.isfinite(cagr) and max_drawdown < 0 else np.nan,
        "Drawdown at final date": float(account.Drawdown.iloc[-1]),
        "Maximum drawdown": max_drawdown,
    })

def run_model(data):
    cash, units, flag = INITIAL_CASH, 0.0, 0
    peak_close = active_stop_pct = np.nan
    records, orders = [], []
    cash_tolerance = max(1.0, INITIAL_CASH) * 1e-12

    for date, row in data.iterrows():
        price, vix_close = float(row.Close), float(row.VIX_Close)
        state, previous = int(row.State), int(row.Previous_state)
        equity_before = cash + units * price
        side, rule_id = None, ""
        target, execution_price = np.nan, np.nan
        commission, slippage_cost = 0.0, 0.0

        # Existing stops take priority over all other signals.
        if units > 0:
            peak_close = max(peak_close, price)
            if price <= peak_close * (1 - active_stop_pct):
                side, rule_id = "SELL", "V1"

        if side is None and units > 0:
            if state == 1 and previous in (2, 3, 4, 5):
                flag = 1
            if flag == 1:
                if previous == 1 and state > 1 and vix_close > RULES["V2"]["vix_min"]:
                    side, rule_id = "SELL", "V2"
                elif state == 1 and vix_close < RULES["V3"]["vix_max"]:
                    side, rule_id = "SELL", "V3"

        if side == "SELL":
            quantity = units
            execution_price = price * (1 - SLIPPAGE_PCT)
            commission = quantity * COMMISSION_PER_SHARE
            slippage_cost = quantity * price * SLIPPAGE_PCT
            cash += quantity * execution_price - commission
            units, flag = 0.0, 0
            peak_close = active_stop_pct = np.nan
        else:
            if previous == 2 and state == 3 and vix_close > RULES["C2"]["vix_min"]:
                rule_id = "C2"
                target = RULES[rule_id]["target_nav"] if units > 0 else RULES[rule_id]["cash_fraction"]
            elif previous == 4 and state == 3 and vix_close < RULES["C3"]["vix_max"]:
                rule_id = "C3"
                target = RULES[rule_id]["invested_cash_fraction"] if units > 0 else RULES[rule_id]["cash_fraction"]

            if np.isfinite(target):
                execution_price = price * (1 + SLIPPAGE_PCT)
                all_in_price = execution_price + COMMISSION_PER_SHARE
                if units == 0 or rule_id == "C3":
                    quantity = target * cash / all_in_price
                else:
                    # C2 targets position value / NAV after transaction costs.
                    gap = max(0.0, target * equity_before - units * price)
                    quantity = min(cash / all_in_price,
                        gap / (price + target * (price * SLIPPAGE_PCT + COMMISSION_PER_SHARE)))

                if quantity * all_in_price > cash_tolerance:
                    side = "BUY"
                    commission = quantity * COMMISSION_PER_SHARE
                    slippage_cost = quantity * price * SLIPPAGE_PCT
                    cash -= quantity * execution_price + commission
                    units += quantity
                    active_stop_pct = RULES[rule_id]["trailing_stop_pct"]
                    peak_close = price

        # Remove floating-point residue without allowing borrowing.
        if -cash_tolerance <= cash < 0:
            cash = 0.0
        if cash < 0 or units < 0:
            raise ArithmeticError(f"Negative cash or units at {date}.")
        if units == 0:
            flag = 0
        stop = peak_close * (1 - active_stop_pct) if units > 0 else np.nan

        if side is not None:
            orders.append({
                "Date": date, "Side": side, "Rule ID": rule_id,
                "Close": price, "Execution price": execution_price, "Units": quantity,
                "Commission ($)": commission, "Slippage ($)": slippage_cost,
                "Cash after": cash, "Units after": units,
            })
        records.append({
            "Date": date, "Close": price, "VIX_Close": vix_close,
            "State": state, "Previous_state": previous,
            "Cash": cash, "Units": units, "Commission": commission,
            "Slippage_cost": slippage_cost, "Peak_close": peak_close,
            "Stop_threshold": stop, "Trailing_stop_pct": active_stop_pct,
            "Flag": flag, "Action": side or "HOLD", "Rule ID": rule_id,
        })

    account = finish_account(pd.DataFrame(records).set_index("Date"))
    order_columns = ["Date", "Side", "Rule ID", "Close", "Execution price", "Units",
                     "Commission ($)", "Slippage ($)", "Cash after", "Units after"]
    return account, pd.DataFrame(orders, columns=order_columns)

## 3. Monte Carlo simulations

In each of 200 simulations, we add independent uniform noise between `−(High−Low)/26` and `+(High−Low)/26` to each VIX Close, using that same day's range. We recalculate states and previous states, including the preceding VIX observation used for the first session. The divisor 26 represents the number of 15-minute intervals in a 6.5-hour session.

The generator starts with seed **2073** whenever the cell runs, so the results can be reproduced.

In [3]:
def metrics(close):
    account, _ = run_model(aligned_prices(close))
    values = performance(account)
    return {
        "CAGR": float(values["Annualized return (CAGR)"]),
        "Sharpe": float(values["Sharpe ratio"]),
        "Calmar": float(values["Calmar ratio"]),
        "Maximum drawdown": abs(float(values["Maximum drawdown"])),
    }

# Reset once per execution, before generating the 200 different paths.
rng = np.random.default_rng(SEED)
noise = rng.uniform(-amplitude, amplitude, size=(N_SIMULATIONS, len(vix)))
baseline = pd.Series(metrics(original_close))
simulations = pd.DataFrame([metrics(original_close + epsilon) for epsilon in noise])
simulations.index = pd.RangeIndex(1, N_SIMULATIONS + 1, name="Run")


## 4. Results

The percentiles describe results across the 200 simulations. Maximum drawdown is reported as a positive magnitude.

In [4]:
summary = pd.DataFrame({
    "Original": baseline,
    "Mean": simulations.mean(),
    "Median": simulations.median(),
    "5th percentile": simulations.quantile(0.05),
    "95th percentile": simulations.quantile(0.95),
}).rename_axis("Metric")

table = summary.astype(object)
for metric in ["CAGR", "Maximum drawdown"]:
    table.loc[metric] = table.loc[metric].map(lambda x: f"{x:.2%}")
for metric in ["Sharpe", "Calmar"]:
    table.loc[metric] = table.loc[metric].map(lambda x: f"{x:.3f}")

styled = (table.style.set_uuid("vix_sensitivity")
    .set_caption("Model 2073 Results with Small Variations in VIX Close")
    .set_table_styles([
        {"selector": "caption", "props": [("text-align", "left"),
            ("color", "#255c80"), ("font-size", "16px"), ("padding", "8px 0")]},
        {"selector": "th", "props": [("background-color", "#255c80"),
            ("color", "white"), ("text-align", "left"), ("padding", "9px 12px")]},
        {"selector": "td", "props": [("padding", "9px 12px"), ("text-align", "right")]},
        {"selector": "tbody tr:nth-child(even)",
         "props": [("background-color", "#edf3f7")]},
        {"selector": "table", "props": [("border-collapse", "collapse")]},
    ]))
display({"text/html": styled.to_html(), "text/plain": table.to_string()}, raw=True)


,Original,Mean,Median,5th percentile,95th percentile
Metric,,,,,
CAGR,23.69%,22.12%,23.17%,16.39%,26.04%
Sharpe,0.914,0.911,0.920,0.753,1.025
Calmar,0.752,0.708,0.733,0.512,0.843
Maximum drawdown,31.49%,31.33%,30.89%,27.41%,38.59%


Average CAGR was 1.57 percentage points lower than in the original run, and average Calmar was also lower. Average Sharpe and maximum drawdown remained close to the original results. However, some simulations were more affected; the lowest CAGR was 12.03%.

The daily range divided by 26 is an arithmetic reference, and the uniform distribution is a simulation assumption. This analysis does not test actual execution before the close; that would require intraday VIX and SVXY data.